# Week 8 Video 4: Random Forest Implementation

## Predicting Diabetes with Random Forest

In this notebook, we'll implement a Random Forest classifier to predict diabetes using the Pima Indians dataset. We'll cover:

1. **Import Libraries** - Loading the necessary tools
2. **Load & Explore Data** - Understanding our dataset
3. **Visualize Data** - Exploring feature relationships
4. **Train-Test Split** - Preparing for model training
5. **Train the Model** - Building our Random Forest
6. **Generate Predictions** - Testing on train and test data
7. **Evaluate Performance** - Measuring accuracy and more
8. **Feature Importance** - Understanding what drives predictions

---
## Step 1: Import Libraries

We need several libraries for this implementation:
- **pandas**: Data manipulation and analysis
- **numpy**: Numerical operations
- **sklearn**: Machine learning tools (model, splitting, metrics)
- **matplotlib/seaborn**: Visualization

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ All libraries imported successfully!")

---
## Step 2: Load and Explore the Dataset

Let's load the Pima Indians Diabetes dataset and understand what we're working with.

In [ ]:
# Load the dataset
diabetes_df = pd.read_csv('diabetes.csv')

# Display basic info
print(f"Dataset Shape: {diabetes_df.shape}")
print(f"\nRows: {diabetes_df.shape[0]}, Columns: {diabetes_df.shape[1]}")
print("\n" + "="*50)
print("First 5 rows:")
diabetes_df.head()

In [ ]:
# Check class distribution
print("Class Distribution:")
print("-" * 30)
class_counts = diabetes_df['Outcome'].value_counts()
print(f"No Diabetes (0): {class_counts[0]} patients ({class_counts[0]/len(diabetes_df)*100:.1f}%)")
print(f"Diabetes (1):    {class_counts[1]} patients ({class_counts[1]/len(diabetes_df)*100:.1f}%)")

print("\n" + "="*50)
print("Dataset Statistics:")
diabetes_df.describe().round(2)

---
## Step 3: Visualize the Data

Before building our model, let's explore the relationships between features using visualizations.

In [ ]:
# Pairplot for key features
key_features = ['Glucose', 'BMI', 'Age', 'DiabetesPedigreeFunction', 'Outcome']

pairplot = sns.pairplot(
    diabetes_df[key_features],
    hue='Outcome',
    palette={0: '#2ecc71', 1: '#e74c3c'},
    diag_kind='kde',
    plot_kws={'alpha': 0.6},
    corner=True
)
pairplot.figure.suptitle('Diabetes Dataset Pairplot', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('rf_pairplot.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n📊 Pairplot saved as 'rf_pairplot.png'")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = diabetes_df.corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='RdYlGn',
    center=0,
    fmt='.2f',
    square=True,
    linewidths=0.5
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_correlation.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Correlation with target
print("\nCorrelation with Diabetes Outcome:")
print("-" * 40)
outcome_corr = correlation_matrix['Outcome'].drop('Outcome').sort_values(ascending=False)
for feature, corr in outcome_corr.items():
    print(f"{feature:30} {corr:+.3f}")

print("\n📊 Correlation heatmap saved as 'rf_correlation.png'")

---
## Step 4: Train-Test Split

We split the data into training (80%) and testing (20%) sets to evaluate model performance on unseen data.

In [ ]:
# Separate features and target
X = diabetes_df.drop('Outcome', axis=1)
y = diabetes_df['Outcome']

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\n✓ Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.0f}%)")
print(f"✓ Testing set:  {len(X_test)} samples ({len(X_test)/len(X)*100:.0f}%)")

---
## Step 5: Train the Random Forest Model

Random Forest combines multiple decision trees to make predictions. We'll use 100 trees (n_estimators).

In [ ]:
# Create and train Random Forest classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest model...")
rf_model.fit(X_train, y_train)
print("✓ Model trained successfully!")

---
## Step 6: Generate Predictions

We generate predictions on both training and test data to compare performance.

In [ ]:
# Generate predictions
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

print("✓ Predictions generated!")
print(f"  Training predictions: {len(y_train_pred)} samples")
print(f"  Test predictions:     {len(y_test_pred)} samples")

# Sample comparison
print("\n" + "="*50)
print("Sample Predictions vs Actual (first 10):")
print("="*50)
comparison_df = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_test_pred[:10],
    'Correct': ['✓' if a == p else '✗' for a, p in zip(y_test.values[:10], y_test_pred[:10])]
})
print(comparison_df.to_string(index=False))

---
## Step 7: Evaluate Model Performance

We evaluate using confusion matrices and classification reports to understand accuracy, precision, recall, and F1-score.

In [ ]:
# Training data evaluation
print("=" * 60)
print("TRAINING DATA PERFORMANCE")
print("=" * 60)

cm_train = confusion_matrix(y_train, y_train_pred)
print("\nConfusion Matrix (Training):")
print(cm_train)

print("\nClassification Report (Training):")
print(classification_report(y_train, y_train_pred, target_names=['No Diabetes', 'Diabetes']))

# Test data evaluation
print("\n" + "=" * 60)
print("TEST DATA PERFORMANCE")
print("=" * 60)

cm_test = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix (Test):")
print(cm_test)

print("\nClassification Report (Test):")
print(classification_report(y_test, y_test_pred, target_names=['No Diabetes', 'Diabetes']))

In [ ]:
# Visualize confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training confusion matrix
ConfusionMatrixDisplay(
    cm_train,
    display_labels=['No Diabetes', 'Diabetes']
).plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Training Data', fontsize=12, fontweight='bold')

# Test confusion matrix
ConfusionMatrixDisplay(
    cm_test,
    display_labels=['No Diabetes', 'Diabetes']
).plot(ax=axes[1], cmap='Oranges', values_format='d')
axes[1].set_title('Test Data', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('rf_confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Calculate accuracies
train_accuracy = (y_train == y_train_pred).mean()
test_accuracy = (y_test == y_test_pred).mean()

print("\n" + "="*50)
print("ACCURACY SUMMARY:")
print("="*50)
print(f"Training Accuracy: {train_accuracy:.1%}")
print(f"Test Accuracy:     {test_accuracy:.1%}")
print(f"Accuracy Gap:      {(train_accuracy - test_accuracy):.1%}")

print("\n📊 Confusion matrices saved as 'rf_confusion_matrices.png'")

---
## Step 8: Feature Importance Analysis

One key advantage of Random Forest is that it tells us which features contribute most to predictions.

In [ ]:
# Extract and display feature importances
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance Ranking:")
print("=" * 50)
for i, row in feature_importance.iterrows():
    bar = '█' * int(row['Importance'] * 50)
    print(f"{row['Feature']:30} {row['Importance']:.3f} {bar}")

In [ ]:
# Visualize feature importance
plt.figure(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(feature_importance)))

bars = plt.barh(
    feature_importance['Feature'],
    feature_importance['Importance'],
    color=colors[::-1]
)

plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Random Forest Feature Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

# Add value labels
for bar, importance in zip(bars, feature_importance['Importance']):
    plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{importance:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n📊 Feature importance plot saved as 'rf_feature_importance.png'")

---
## Summary: Key Takeaways

### What We Learned:

1. **Random Forest = Ensemble of Trees** - Combines 100 decision trees for better accuracy

2. **No Scaling Required** - Unlike k-NN, Random Forest doesn't need feature scaling

3. **Feature Importance** - Glucose is the strongest predictor of diabetes

4. **Overfitting Risk** - Perfect training accuracy vs lower test accuracy indicates overfitting

5. **Interpretability** - We can understand which features drive predictions

### Model Results:
- **Training Accuracy**: 100% (model memorized training data)
- **Test Accuracy**: ~76% (real-world performance)
- **Top Features**: Glucose, BMI, Age

In [ ]:
# Final summary
print("\n" + "="*60)
print("🎓 WEEK 8-4: RANDOM FOREST IMPLEMENTATION COMPLETE!")
print("="*60)
print(f"\n✓ Number of trees: 100")
print(f"✓ Test Accuracy: {test_accuracy:.1%}")
print(f"✓ Train-Test Gap: {(train_accuracy - test_accuracy):.1%} (indicates overfitting)")
print(f"✓ Top Feature: {feature_importance.iloc[0]['Feature']} ({feature_importance.iloc[0]['Importance']:.1%} importance)")
print("\n📁 Generated files:")
print("   • rf_pairplot.png")
print("   • rf_correlation.png")
print("   • rf_confusion_matrices.png")
print("   • rf_feature_importance.png")